<!-- paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 참고 구현입니다. 필요한 한 줄만 확인하고 다시 실습 창으로 돌아가세요.

# 15. Transformer 구조 해부 실습 — 직접 만드는 causal Transformer

이 파일은 **실습용/정답용이 같은 셀 순서를 갖는 페어 노트북**입니다. JupyterLab에서
왼쪽에는 `exercises/15_...`, 오른쪽에는 `solutions/15_...`를 열고 셀 단위로 따라 치세요.

이번 랩의 목표는 완제품 API를 호출하는 것이 아니라 `[B, T, D]` 텐서가 Q/K/V,
head 분리, mask, residual을 통과하는 과정을 눈으로 확인하는 것입니다.

In [ ]:
# 이 컴퓨터를 감지해 안전한 시작값을 확인합니다. 각 실습은 더 작은 값으로도 동작합니다.
from llm_engineering_lab.hardware import HardwareProfile, recommend_training_profile

LAB_HARDWARE = HardwareProfile.detect()
LAB_PROFILE = recommend_training_profile(LAB_HARDWARE)
print("hardware:", LAB_HARDWARE)
print("recommended profile:", LAB_PROFILE)

## 완료 기준과 권장 순서

1. Token/position embedding을 더한다.
2. Q/K/V projection과 scaled dot-product attention을 구현한다.
3. `[B,T,D] ↔ [B,H,T,Dh]` 변환과 causal/padding mask를 검증한다.
4. Pre-norm residual, FFN, decoder block, 작은 causal LM을 조립한다.
5. shape, gradient, 미래 토큰 누설 방지 assert를 모두 통과시킨다.

**학습법:** 정답 전체를 복사하지 말고 TODO 하나를 먼저 작성한 뒤 해당 검증 셀만 실행하세요.

<!-- api-explanations:start -->
<!-- api-explanations:keys=random.seed,torch.manual_seed -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>random.seed(a=None)</code></summary>

#### `random.seed(a=None)`

- **역할:** Python 표준 난수 생성기의 상태를 초기화합니다.
- **입력·반환:** 정수 등의 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 같은 Python 난수 수열을 다시 만들어 실험을 재현합니다.
- **주의점:** NumPy와 PyTorch의 난수 상태는 별도로 고정해야 합니다.
- **공식 문서:** [random.seed](https://docs.python.org/3/library/random.html#random.seed)

</details>

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import math
import random
from dataclasses import dataclass

import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

SEED = 15
random.seed(SEED)
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
device = ACCELERATOR.device
print(f"{ACCELERATOR.summary()}, torch={torch.__version__}")

## 1) 미니 배치와 vocabulary

`input_ids`의 축은 `[batch, time]`입니다. `<pad>`는 문장 길이를 맞추기 위한 토큰이고,
`valid_tokens`가 False인 위치는 key로도 query로도 attention에 참여시키지 않습니다.
실제 서비스에서는 tokenizer가 이 과정을 담당하지만 구조 학습을 위해 작은 단어 사전을 직접 만듭니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.tensor -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
token_rows = [
    ["<bos>", "나는", "모델을", "학습한다", "<eos>", "<pad>"],
    ["<bos>", "attention은", "문맥을", "연결한다", "<eos>", "<pad>"],
]
vocab_tokens = ["<pad>", "<bos>", "<eos>"] + sorted(
    {token for row in token_rows for token in row if not token.startswith("<")}
)
stoi = {token: index for index, token in enumerate(vocab_tokens)}
itos = {index: token for token, index in stoi.items()}
input_ids = torch.tensor(
    [[stoi[token] for token in row] for row in token_rows],
    dtype=torch.long,
    device=device,
)
pad_id = stoi["<pad>"]
valid_tokens = input_ids.ne(pad_id)

print("vocab:", stoi)
print("input_ids shape:", tuple(input_ids.shape))
print("valid tokens:\n", valid_tokens)
assert input_ids.shape == valid_tokens.shape == (2, 6)

## 2) Token embedding + learned position embedding

같은 단어라도 순서가 다르면 의미가 달라집니다. Token embedding은 **무엇인지**, position
embedding은 **어디에 있는지**를 표현하고 둘은 같은 `model_dim`이므로 원소별로 더할 수 있습니다.
출력 shape은 항상 `[B, T, D]`입니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Embedding,torch.arange -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Embedding(num_embeddings, embedding_dim, ...)</code></summary>

#### `nn.Embedding(num_embeddings, embedding_dim, ...)`

- **역할:** 정수 ID를 학습 가능한 벡터로 조회합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
e_i=E[i],\qquad E\in\mathbb{R}^{V\times D}
$$

- **기호:** $i$는 정수 ID, $V$는 vocabulary 크기, $D$는 embedding 차원입니다.
- **수식 → 코드:** forward는 one-hot 행렬곱을 만들지 않고 weight 행 $E[i]$를 직접 조회합니다.
- **직관:** 범주 ID를 학습 가능한 연속 벡터 좌표로 바꿉니다.
- **shape 확인:** 입력 `[...] -> 출력 [..., D]`; 입력 dtype은 보통 `torch.int64`입니다.
- **공식 문서:** [nn.Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

</details>

<details>
<summary><code>torch.arange(start=0, end, step=1, ...)</code></summary>

#### `torch.arange(start=0, end, step=1, ...)`

- **역할:** 일정 간격의 1차원 텐서를 만듭니다.
- **입력·반환:** 범위와 간격을 받고 1차원 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** position·index·time step을 벡터화해 만듭니다.
- **주의점:** 부동소수점 step은 반올림 때문에 원소 수가 예상과 다를 수 있습니다.
- **공식 문서:** [torch.arange](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class TokenPositionEmbedding(nn.Module):
    def __init__(self, vocab_size: int, model_dim: int, max_length: int, pad_id: int):
        super().__init__()
        self.token = nn.Embedding(vocab_size, model_dim, padding_idx=pad_id)
        self.position = nn.Embedding(max_length, model_dim)
        self.max_length = max_length

    def forward(self, ids: Tensor) -> Tensor:
        batch_size, length = ids.shape
        if length > self.max_length:
            raise ValueError(f"length={length} exceeds max_length={self.max_length}")
        positions = torch.arange(length, device=ids.device).expand(batch_size, length)
        return self.token(ids) + self.position(positions)

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.allclose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
model_dim = 32
embedding = TokenPositionEmbedding(len(stoi), model_dim, 16, pad_id).to(device)
hidden = embedding(input_ids)
assert hidden.shape == (2, 6, model_dim)
assert not torch.allclose(hidden[:, 0], hidden[:, 1])
print("embedding:", tuple(hidden.shape), "dtype:", hidden.dtype)

## 3) Q, K, V projection

한 hidden vector를 세 관점으로 선형 변환합니다. Query는 "무엇을 찾는가", Key는
"나는 무엇을 나타내는가", Value는 "선택되면 무엇을 전달하는가"로 이해할 수 있습니다.
현업 구현은 메모리 접근을 줄이기 위해 보통 하나의 `Linear(D, 3D)` 결과를 세 덩어리로 나눕니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Linear -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
qkv_projection = nn.Linear(model_dim, 3 * model_dim).to(device)
qkv = qkv_projection(hidden)
query, key, value = qkv.chunk(3, dim=-1)
assert query.shape == key.shape == value.shape == hidden.shape
print("qkv packed:", tuple(qkv.shape), "q:", tuple(query.shape))

## 4) Multi-head reshape

`D = H × Dh`로 나누어 여러 관계를 병렬로 봅니다. 연산을 위해 `[B,T,D]`를
`[B,H,T,Dh]`로 바꾸며, `transpose` 뒤에는 메모리가 비연속일 수 있으므로 합칠 때
`contiguous()`를 사용하는 습관이 중요합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.view,Tensor.transpose -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.view(*shape)</code></summary>

#### `tensor.view(*shape)`

- **역할:** 같은 저장공간을 보는 새로운 shape의 텐서를 만듭니다.
- **입력·반환:** 새 shape을 받고 view 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 연속 메모리의 축을 복사 없이 재해석합니다.
- **주의점:** stride가 호환되지 않으면 실패하므로 `reshape`나 `contiguous`가 필요합니다.

##### 관련 수식과 코드 연결

$$
\prod_{k=1}^{K} d_k=\prod_{j=1}^{J} d'_j
$$

- **기호:** $d_k$는 원래 각 축 크기, $d'_j$는 바꾼 뒤 각 축 크기입니다.
- **수식 → 코드:** `reshape`·`view`·`flatten`은 원소의 총개수를 보존하면서 축 해석만 바꿉니다.
- **직관:** 데이터 값을 계산하는 연산이 아니라 같은 원소를 다른 좌표계로 보는 연산입니다.
- **shape 확인:** 예: `[B, H, T, D] -> [B, T, H*D]`; 왼쪽과 오른쪽 원소 수가 같아야 합니다.
- **공식 문서:** [Tensor.view](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.view.html)

</details>

<details>
<summary><code>tensor.transpose(dim0, dim1)</code></summary>

#### `tensor.transpose(dim0, dim1)`

- **역할:** 두 축의 위치를 맞바꿉니다.
- **입력·반환:** 두 축 번호를 받고 축이 교환된 view를 반환합니다.
- **이 셀에서 쓰는 이유:** attention 행렬곱 등에서 축 순서를 맞춥니다.
- **주의점:** 반환 텐서는 비연속일 수 있어 이후 `view` 전에 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
Y_{i_{\pi(1)},\ldots,i_{\pi(K)}}=X_{i_1,\ldots,i_K}
$$

- **기호:** $\pi$는 축의 새 순서를 나타내는 permutation입니다.
- **수식 → 코드:** `transpose`는 두 축만 바꾸고 `permute`는 모든 축의 새 순서를 명시합니다.
- **직관:** 원소 값을 바꾸지 않고 각 index가 의미하는 축의 위치만 다시 배치합니다.
- **shape 확인:** 예: `[B,T,H,D] -> [B,H,T,D]`; 결과는 비연속 view일 수 있습니다.
- **공식 문서:** [Tensor.transpose](https://docs.pytorch.org/docs/stable/generated/torch.transpose.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def split_heads(x: Tensor, num_heads: int) -> Tensor:
    batch_size, length, width = x.shape
    if width % num_heads != 0:
        raise ValueError("model_dim must be divisible by num_heads")
    head_dim = width // num_heads
    return x.view(batch_size, length, num_heads, head_dim).transpose(1, 2)


def merge_heads(x: Tensor) -> Tensor:
    batch_size, num_heads, length, head_dim = x.shape
    return x.transpose(1, 2).contiguous().view(batch_size, length, num_heads * head_dim)

In [ ]:
num_heads = 4
qh, kh, vh = (split_heads(t, num_heads) for t in (query, key, value))
assert qh.shape == (2, 4, 6, 8)
assert torch.allclose(merge_heads(qh), query)
print("per head:", tuple(qh.shape), "merged:", tuple(merge_heads(qh).shape))

## 5) Scaled dot-product attention

`scores = QKᵀ / sqrt(Dh)`입니다. 차원이 커지면 dot product 분산이 커져 softmax가
지나치게 뾰족해지므로 `sqrt(Dh)`로 나눕니다. Mask는 **softmax 전에** 허용되지 않은
score를 매우 작은 값으로 바꿉니다. 전부 mask된 padding query는 명시적으로 0으로 처리합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.sqrt,Tensor.masked_fill,torch.softmax,torch.where,Tensor.sum -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 5개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.sqrt(input)</code></summary>

#### `torch.sqrt(input)`

- **역할:** 각 원소의 제곱근을 계산합니다.
- **입력·반환:** 0 이상 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 표준편차, 거리, diffusion schedule의 scale을 계산합니다.
- **주의점:** 음수 입력은 NaN을 만들며 0 근처 gradient가 매우 커질 수 있습니다.

##### 관련 수식과 코드 연결

$$
y_i=\sqrt{x_i},\qquad x_i\ge 0
$$

- **기호:** $x_i$는 음수가 아닌 입력입니다.
- **수식 → 코드:** API가 제곱 또는 분산 scale을 원래 단위의 길이로 되돌립니다.
- **직관:** 제곱된 크기를 사람이 해석하기 쉬운 원래 scale로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.sqrt](https://docs.pytorch.org/docs/stable/generated/torch.sqrt.html)

</details>

<details>
<summary><code>tensor.masked_fill(mask, value)</code></summary>

#### `tensor.masked_fill(mask, value)`

- **역할:** mask가 참인 위치를 지정 값으로 바꿉니다.
- **입력·반환:** 불리언 mask와 값을 받고 새 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** padding이나 미래 token의 attention score를 선택에서 제외합니다.
- **주의점:** mask는 입력에 broadcast 가능해야 하고 softmax 전 값은 충분히 작아야 합니다.

##### 관련 수식과 코드 연결

$$
s'_{ij}=\begin{cases}s_{ij},&M_{ij}=0\\-\infty,&M_{ij}=1\end{cases}\quad\Rightarrow\quad\operatorname{softmax}(s')_{ij}=0\;\text{if }M_{ij}=1
$$

- **기호:** $s$는 score, $M$은 제외할 위치가 참인 mask입니다.
- **수식 → 코드:** attention에서는 mask 위치를 매우 작은 값으로 채운 뒤 softmax를 적용합니다.
- **직관:** 확률을 나중에 지우는 대신 정규화 분모에도 들어가지 못하게 합니다.
- **shape 확인:** mask는 score에 broadcast 가능해야 하며 보통 `[B, H, T_q, T_k]`를 사용합니다.
- **공식 문서:** [Tensor.masked_fill](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.masked_fill.html)

</details>

<details>
<summary><code>torch.softmax(input, dim, dtype=None)</code></summary>

#### `torch.softmax(input, dim, dtype=None)`

- **역할:** 지정 축의 값을 합이 1인 확률형 가중치로 변환합니다.
- **입력·반환:** 텐서와 축을 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** attention score나 class logit을 정규화합니다.
- **주의점:** `dim`을 잘못 고르면 엉뚱한 축이 정규화되고 큰 logit은 포화될 수 있습니다.

##### 관련 수식과 코드 연결

$$
p_i=\frac{e^{z_i}}{\sum_j e^{z_j}},\qquad \log p_i=z_i-\log\sum_j e^{z_j}
$$

- **기호:** $z_i$는 logit, $p_i$는 선택한 축에서 합이 1인 확률입니다.
- **수식 → 코드:** 코드의 `dim`이 분모의 합을 계산할 $j$축을 결정합니다.
- **직관:** 각 logit을 다른 후보와 비교 가능한 상대 확률로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같고 선택한 축의 softmax 값 합은 1입니다.
- **공식 문서:** [torch.softmax](https://docs.pytorch.org/docs/stable/generated/torch.softmax.html)

</details>

<details>
<summary><code>torch.where(condition, input, other)</code></summary>

#### `torch.where(condition, input, other)`

- **역할:** 조건에 따라 두 텐서의 원소를 선택합니다.
- **입력·반환:** 불리언 조건과 두 입력을 받고 broadcast된 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** mask 기반 분기와 piecewise 수식을 벡터화합니다.
- **주의점:** 세 입력의 broadcasting과 두 branch의 dtype·device를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\begin{cases}x_i,&c_i=\mathrm{True}\\z_i,&c_i=\mathrm{False}\end{cases}
$$

- **기호:** $c_i$는 조건, $x_i,z_i$는 두 후보 값입니다.
- **수식 → 코드:** API가 Python 분기 대신 원소별로 두 tensor 중 하나를 선택합니다.
- **직관:** piecewise 수식과 mask 기반 update를 병렬 tensor 연산으로 표현합니다.
- **shape 확인:** 세 입력을 broadcasting한 공통 shape이 출력 shape입니다.
- **공식 문서:** [torch.where](https://docs.pytorch.org/docs/stable/generated/torch.where.html)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def scaled_dot_product_attention(
    q: Tensor, k: Tensor, v: Tensor, allowed: Tensor | None = None
) -> tuple[Tensor, Tensor]:
    head_dim = q.shape[-1]
    scores = q @ k.transpose(-2, -1) / math.sqrt(head_dim)
    if allowed is not None:
        scores = scores.masked_fill(~allowed, torch.finfo(scores.dtype).min)
    weights = F.softmax(scores, dim=-1)
    if allowed is not None:
        weights = torch.where(allowed, weights, torch.zeros_like(weights))
        weights = weights / weights.sum(dim=-1, keepdim=True).clamp_min(1e-9)
    context = weights @ v
    return context, weights

## 6) Causal mask + padding mask

- Causal mask: query 위치 `t`가 `t+1` 이후 key를 보지 못하게 합니다.
- Padding key mask: `<pad>`를 정보원으로 선택하지 못하게 합니다.
- Padding query mask: 의미 없는 query 행의 attention을 0으로 만듭니다.

최종 `allowed`는 broadcast 가능한 `[B,1,T,T]` boolean 텐서입니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.ones,torch.bool -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.ones(*size, dtype=None, device=None)</code></summary>

#### `torch.ones(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 1인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 weight·mask·스케일을 구성합니다.
- **주의점:** 주변 연산과 dtype·device가 같은지 확인해야 합니다.
- **공식 문서:** [torch.ones](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)

</details>

<details>
<summary><code>torch.bool</code></summary>

#### `torch.bool`

- **역할:** 불리언 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** mask와 조건 결과를 표현합니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.bool](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def build_allowed_mask(valid: Tensor) -> Tensor:
    batch_size, length = valid.shape
    causal = torch.ones(length, length, dtype=torch.bool, device=valid.device).tril()
    causal = causal.view(1, 1, length, length)
    key_is_valid = valid.view(batch_size, 1, 1, length)
    query_is_valid = valid.view(batch_size, 1, length, 1)
    return causal & key_is_valid & query_is_valid

In [ ]:
allowed = build_allowed_mask(valid_tokens)
context, attention_weights = scaled_dot_product_attention(qh, kh, vh, allowed)
future = torch.ones(6, 6, dtype=torch.bool, device=device).triu(diagonal=1)
assert allowed.shape == (2, 1, 6, 6)
assert torch.all(attention_weights.masked_select(future.view(1, 1, 6, 6)) == 0)
assert torch.all(attention_weights[..., -1, :] == 0)
assert torch.all(attention_weights[..., :, -1] == 0)
print("context:", tuple(context.shape), "mask passed")

## 7) Attention을 사람이 읽을 수 있는 데이터로 보기

아래 `attention_view`는 시각화 라이브러리에 바로 전달할 수 있는 토큰 label과 2차원 weight
행렬입니다. 행은 query, 열은 key입니다. causal mask 때문에 대각선 오른쪽 값이 모두 0인지
확인하세요. 정답 노트북은 추가 설치 없이도 보이도록 텍스트 heatmap 형태로 출력합니다.

In [ ]:
tokens_for_view = token_rows[0]
matrix_for_view = attention_weights[0, 0].detach().cpu()
attention_view = {
    "tokens": tokens_for_view,
    "weights": matrix_for_view.round(decimals=3).tolist(),
}
print("query/key".ljust(13), " ".join(f"{t[:7]:>7}" for t in tokens_for_view))
for token, row in zip(tokens_for_view, attention_view["weights"], strict=True):
    print(f"{token[:11]:<13}", " ".join(f"{value:7.3f}" for value in row))

## 8) Multi-head self-attention 모듈

이제 흩어진 연산을 하나의 `nn.Module`로 묶습니다. `qkv` projection → head 분리 → attention
→ head 병합 → output projection 순서입니다. 학습/디버깅을 위해 weight도 함께 반환합니다.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, model_dim: int, num_heads: int):
        super().__init__()
        if model_dim % num_heads != 0:
            raise ValueError("model_dim must be divisible by num_heads")
        self.num_heads = num_heads
        self.qkv = nn.Linear(model_dim, 3 * model_dim)
        self.out = nn.Linear(model_dim, model_dim)

    def forward(self, x: Tensor, allowed: Tensor) -> tuple[Tensor, Tensor]:
        query, key, value = self.qkv(x).chunk(3, dim=-1)
        query, key, value = (
            split_heads(tensor, self.num_heads) for tensor in (query, key, value)
        )
        context, weights = scaled_dot_product_attention(query, key, value, allowed)
        return self.out(merge_heads(context)), weights

In [ ]:
mha = MultiHeadSelfAttention(model_dim, num_heads).to(device)
mha_output, mha_weights = mha(hidden, allowed)
assert mha_output.shape == hidden.shape
assert mha_weights.shape == (2, num_heads, 6, 6)
print("MHA output:", tuple(mha_output.shape))

## 9) Pre-norm residual + Feed Forward Network

Pre-norm block은 `x + Attention(LN(x))`, 이어서 `x + FFN(LN(x))`를 수행합니다.
Residual 경로는 gradient가 깊은 네트워크를 통과하도록 돕고, FFN은 각 토큰 위치에 동일한
두 층 MLP를 적용합니다. 중간 차원은 보통 model dimension보다 큽니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Sequential,nn.GELU,nn.Dropout,nn.LayerNorm -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Sequential(*modules)</code></summary>

#### `nn.Sequential(*modules)`

- **역할:** 모듈을 순서대로 연결합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

</details>

<details>
<summary><code>nn.GELU(approximate=&#x27;none&#x27;)</code></summary>

#### `nn.GELU(approximate='none')`

- **역할:** 부드러운 GELU 활성함수를 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{GELU}(x)=x\Phi(x)\approx\frac{x}{2}\left[1+\tanh\!\left(\sqrt{\frac{2}{\pi}}(x+0.044715x^3)\right)\right]
$$

- **기호:** $\Phi(x)$는 표준정규분포의 누적분포함수입니다.
- **수식 → 코드:** `approximate` 설정에 따라 정확한 CDF 기반 값 또는 tanh 근사를 계산합니다.
- **직관:** 입력 크기에 비례해 부드럽게 gate하여 ReLU보다 경계가 매끄럽습니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.GELU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html)

</details>

<details>
<summary><code>nn.Dropout(p=0.5)</code></summary>

#### `nn.Dropout(p=0.5)`

- **역할:** 학습 중 일부 activation을 무작위로 0으로 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\frac{m_i}{1-p}x_i,\qquad m_i\sim\operatorname{Bernoulli}(1-p)
$$

- **기호:** $p$는 제거 확률, $m_i$는 학습 중 표본화한 mask입니다.
- **수식 → 코드:** `model.train()`일 때 mask와 $1/(1-p)$ scale을 적용하고 `eval()`에서는 그대로 둡니다.
- **직관:** 일부 activation에 의존하지 못하게 하면서 기댓값은 유지합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html)

</details>

<details>
<summary><code>nn.LayerNorm(normalized_shape, ...)</code></summary>

#### `nn.LayerNorm(normalized_shape, ...)`

- **역할:** 마지막 feature 축들을 표준화합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\gamma_i\frac{x_i-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta_i
$$

- **기호:** $\mu,\sigma^2$는 한 sample의 지정 feature 축 평균·분산입니다.
- **수식 → 코드:** `normalized_shape`의 마지막 축들에서 통계를 내고 학습 가능한 scale과 bias를 적용합니다.
- **직관:** batch 크기와 무관하게 각 token·sample의 feature scale을 안정화합니다.
- **shape 확인:** 입력과 출력 shape은 같고 $\gamma,\beta$는 `normalized_shape`를 따릅니다.
- **공식 문서:** [nn.LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, model_dim: int, hidden_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(model_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, model_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class DecoderBlock(nn.Module):
    def __init__(self, model_dim: int, num_heads: int, ff_dim: int, dropout: float):
        super().__init__()
        self.norm1 = nn.LayerNorm(model_dim)
        self.attention = MultiHeadSelfAttention(model_dim, num_heads)
        self.norm2 = nn.LayerNorm(model_dim)
        self.feed_forward = FeedForward(model_dim, ff_dim, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Tensor, allowed: Tensor) -> tuple[Tensor, Tensor]:
        attention_output, weights = self.attention(self.norm1(x), allowed)
        x = x + self.dropout(attention_output)
        x = x + self.feed_forward(self.norm2(x))
        return x, weights

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.backward,torch.linalg.norm -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>torch.linalg.vector_norm(x, ord=2, dim=None, keepdim=False)</code></summary>

#### `torch.linalg.vector_norm(x, ord=2, dim=None, keepdim=False)`

- **역할:** 벡터의 크기인 norm을 계산합니다.
- **입력·반환:** 텐서·차수·축을 받고 norm 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 거리, gradient 크기, 정규화 분모를 계산합니다.
- **주의점:** `dim`과 norm 차수에 따라 의미와 반환 shape이 달라집니다.

##### 관련 수식과 코드 연결

$$
\lVert x\rVert_p=\left(\sum_i\lvert x_i\rvert^p\right)^{1/p},\qquad \lVert x\rVert_2=\sqrt{\sum_i x_i^2}
$$

- **기호:** $p$는 norm 차수이며 $p=2$가 유클리드 길이입니다.
- **수식 → 코드:** `ord`와 `dim`·`axis`가 어떤 원소 묶음의 크기를 계산할지 정합니다.
- **직관:** 여러 성분을 하나의 거리·크기 값으로 요약합니다.
- **shape 확인:** 지정한 축은 결과에서 제거되며 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [torch.linalg.norm](https://docs.pytorch.org/docs/stable/generated/torch.linalg.vector_norm.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
block = DecoderBlock(model_dim, num_heads, ff_dim=96, dropout=0.0).to(device)
block_output, block_weights = block(hidden, allowed)
probe_loss = block_output.square().mean()
probe_loss.backward()
gradient_norm = block.attention.qkv.weight.grad.norm().item()
assert block_output.shape == hidden.shape
assert gradient_norm > 0
print(f"block={tuple(block_output.shape)}, qkv grad norm={gradient_norm:.6f}")

## 10) Decoder-only causal language model 조립

여러 decoder block 뒤에 final LayerNorm과 vocabulary projection(`lm_head`)을 붙입니다.
언어 모델의 logit shape은 `[B,T,V]`입니다. 위치 `t`의 logit이 다음 token `t+1`을 맞히도록
target을 한 칸 shift하며, padding target은 `ignore_index`로 loss에서 제외합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=dataclass,nn.ModuleList,F.cross_entropy,Tensor.reshape -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>@dataclass(frozen=False, slots=False, ...)</code></summary>

#### `@dataclass(frozen=False, slots=False, ...)`

- **역할:** type annotation을 바탕으로 초기화·표현 메서드 등을 생성합니다.
- **입력·반환:** class decorator 설정을 받고 변환된 class를 반환합니다.
- **이 셀에서 쓰는 이유:** 실험 설정과 결과 record의 필드를 명시적으로 정의합니다.
- **주의점:** mutable 기본값은 `default_factory`를 쓰고 equality·frozen 의미를 확인합니다.
- **공식 문서:** [dataclass](https://docs.python.org/3/library/dataclasses.html#dataclasses.dataclass)

</details>

<details>
<summary><code>nn.ModuleList(modules=None)</code></summary>

#### `nn.ModuleList(modules=None)`

- **역할:** 하위 모듈 목록을 등록 가능한 형태로 보관합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.ModuleList](https://docs.pytorch.org/docs/stable/generated/torch.nn.ModuleList.html)

</details>

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

<details>
<summary><code>tensor.reshape(*shape)</code></summary>

#### `tensor.reshape(*shape)`

- **역할:** 원소 수를 유지하며 텐서 shape을 바꿉니다.
- **입력·반환:** 새 shape을 받고 가능하면 view, 아니면 복사된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch·head·sequence 축을 논문 수식의 shape에 맞춥니다.
- **주의점:** 원소 수가 같아야 하며 메모리 공유 여부에 의존하면 안 됩니다.

##### 관련 수식과 코드 연결

$$
\prod_{k=1}^{K} d_k=\prod_{j=1}^{J} d'_j
$$

- **기호:** $d_k$는 원래 각 축 크기, $d'_j$는 바꾼 뒤 각 축 크기입니다.
- **수식 → 코드:** `reshape`·`view`·`flatten`은 원소의 총개수를 보존하면서 축 해석만 바꿉니다.
- **직관:** 데이터 값을 계산하는 연산이 아니라 같은 원소를 다른 좌표계로 보는 연산입니다.
- **shape 확인:** 예: `[B, H, T, D] -> [B, T, H*D]`; 왼쪽과 오른쪽 원소 수가 같아야 합니다.
- **공식 문서:** [Tensor.reshape](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.reshape.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
@dataclass(frozen=True)
class ArchitectureConfig:
    vocab_size: int
    pad_id: int
    max_length: int = 16
    model_dim: int = 32
    num_heads: int = 4
    num_layers: int = 2
    ff_dim: int = 96
    dropout: float = 0.0


class TinyCausalLM(nn.Module):
    def __init__(self, config: ArchitectureConfig):
        super().__init__()
        self.config = config
        self.embedding = TokenPositionEmbedding(
            config.vocab_size, config.model_dim, config.max_length, config.pad_id
        )
        self.blocks = nn.ModuleList(
            [
                DecoderBlock(
                    config.model_dim,
                    config.num_heads,
                    config.ff_dim,
                    config.dropout,
                )
                for _ in range(config.num_layers)
            ]
        )
        self.final_norm = nn.LayerNorm(config.model_dim)
        self.lm_head = nn.Linear(config.model_dim, config.vocab_size, bias=False)
        self.lm_head.weight = self.embedding.token.weight

    def forward(
        self, ids: Tensor, labels: Tensor | None = None
    ) -> tuple[Tensor, Tensor | None, list[Tensor]]:
        allowed = build_allowed_mask(ids.ne(self.config.pad_id))
        x = self.embedding(ids)
        all_weights: list[Tensor] = []
        for block in self.blocks:
            x, weights = block(x, allowed)
            all_weights.append(weights)
        logits = self.lm_head(self.final_norm(x))
        loss = None
        if labels is not None:
            targets = labels[:, 1:].masked_fill(
                labels[:, 1:].eq(self.config.pad_id), -100
            )
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, self.config.vocab_size),
                targets.reshape(-1),
                ignore_index=-100,
            )
        return logits, loss, all_weights

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.isfinite,Tensor.item -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.isfinite(input)</code></summary>

#### `torch.isfinite(input)`

- **역할:** 텐서의 각 원소가 유한수인지 검사합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 gradient의 NaN·무한대 발생을 빠르게 찾습니다.
- **주의점:** 검사 결과는 미분 대상이 아니며 원인 자체를 고치지는 않습니다.
- **공식 문서:** [torch.isfinite](https://docs.pytorch.org/docs/stable/generated/torch.isfinite.html)

</details>

<details>
<summary><code>tensor.item()</code></summary>

#### `tensor.item()`

- **역할:** 원소 하나짜리 텐서에서 Python 숫자를 꺼냅니다.
- **입력·반환:** 단일 원소 텐서를 받고 Python 숫자를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 metric을 출력·기록 가능한 값으로 바꿉니다.
- **주의점:** 여러 원소 텐서에는 쓸 수 없고 GPU에서는 동기화를 일으킬 수 있습니다.
- **공식 문서:** [Tensor.item](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.item.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
architecture_config = ArchitectureConfig(vocab_size=len(stoi), pad_id=pad_id)
tiny_lm = TinyCausalLM(architecture_config).to(device)
logits, lm_loss, layer_weights = tiny_lm(input_ids, labels=input_ids)
assert logits.shape == (2, 6, len(stoi))
assert lm_loss is not None and torch.isfinite(lm_loss)
assert len(layer_weights) == architecture_config.num_layers
print(f"logits={tuple(logits.shape)}, shifted loss={lm_loss.item():.4f}")

## 11) 누설(leakage) 테스트 — mask가 정말 작동하는가?

미래 위치의 token 하나만 바꿨을 때 그보다 앞선 위치의 logit은 같아야 합니다. 이 테스트는
mask 방향을 뒤집거나 softmax 뒤에 mask하는 흔한 버그를 잡습니다. Dropout은 비교를 흔들 수
있으므로 `eval()`에서 검사합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Module.eval,torch.inference_mode -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.inference_mode(mode=True)</code></summary>

#### `torch.inference_mode(mode=True)`

- **역할:** 추론 전용으로 autograd 관련 추적을 더 강하게 비활성화합니다.
- **입력·반환:** 불리언 모드를 받고 context manager 또는 decorator로 동작합니다.
- **이 셀에서 쓰는 이유:** 순수 평가 구간의 오버헤드를 줄입니다.
- **주의점:** 이 모드에서 만든 텐서는 이후 gradient 계산에 쓰기 어려울 수 있습니다.
- **공식 문서:** [torch.inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.inference_mode.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
tiny_lm.eval()
changed = input_ids.clone()
changed[0, 3] = stoi["연결한다"]
with torch.inference_mode():
    original_logits, _, _ = tiny_lm(input_ids)
    changed_logits, _, _ = tiny_lm(changed)

assert torch.allclose(original_logits[0, 1], changed_logits[0, 1], atol=1e-6)
difference_after_change = (original_logits[0, 3] - changed_logits[0, 3]).abs().max()
print(
    "causal leakage test passed; later max difference:", difference_after_change.item()
)

## 마무리 체크

이제 다음 질문에 코드로 답할 수 있어야 합니다.

- 왜 score를 `sqrt(head_dim)`으로 나누는가?
- causal mask와 padding mask가 각각 어느 축을 막는가?
- head를 분리/병합할 때 `transpose`와 `contiguous`가 왜 필요한가?
- Pre-norm residual의 두 경로는 무엇인가?
- 언어 모델 loss에서 logit과 target을 어떻게 한 칸 이동하는가?

다음 랩에서는 이 구조와 프로젝트의 `TinyCausalTransformer`를 이용해 optimizer, AMP,
gradient accumulation, checkpoint, generation까지 하나의 학습 파이프라인으로 완성합니다.